# 01 · Data: splits, masks, backgrounds
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/01_data.ipynb)

Неделя 1 из `PLAN.md`:
1. Stratified train/val split из `trainval`, официальный `test` без изменений.
2. Аудит изображений и масок.
3. Пулы фонов из Places365 (train / val / test — разные категории сцен).
4. Фиксированные фоны для val и test (manifest).
5. Галерея «оригинал → маска → новый фон → нейтральный фон → только фон».
6. Smoke test DataLoader с заменой фона p = 0.5.

Результаты сохраняются в репозиторий (`splits/`, `manifests/`, `results/`, `report/figures/`) и в `MyDrive/bgbias/week1/`.

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
from bgbias.data_setup import prepare_pets, prepare_places
from bgbias.utils import DRIVE_ROOT, LOCAL_DATA

PET_ROOT = prepare_pets()        # /content/data/oxford-iiit-pet
PLACES_ROOT = prepare_places()   # /content/data/places365
OUT_DRIVE = DRIVE_ROOT / "week1"
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
print(PET_ROOT, PLACES_ROOT)

## 1. Splits

In [ ]:
from pathlib import Path
from bgbias import splits as S

SEED = 42
VAL_FRAC = 0.15
splits = S.make_splits(PET_ROOT, val_frac=VAL_FRAC, seed=SEED)
S.save_splits(splits, Path(REPO_DIR) / "splits")
for name, df in splits.items():
    print(f"{name:5s} {len(df):5d} images | {df.label.nunique()} breeds | "
          f"cats {(df.species == 'cat').sum()}, dogs {(df.species == 'dog').sum()}")

In [ ]:
table = S.class_table(splits)
table["total"] = table[["train", "val", "test"]].sum(axis=1)
print("min / max images per breed in train:", table.train.min(), table.train.max())
table

## 2. Аудит изображений и масок

In [ ]:
import pandas as pd
all_ids = pd.concat(splits.values()).image_id
audit = S.audit_images(PET_ROOT, all_ids)
audit.to_csv(Path(REPO_DIR) / "results" / "data_audit.csv", index=False)

print("modes:", audit["mode"].value_counts().to_dict())
print("broken images:", (~audit.image_ok).sum(), "| broken / mismatched trimaps:", (~audit.trimap_ok).sum())
print(audit[["fg_frac", "bg_frac", "border_frac"]].describe().round(3))
problems = audit[~audit.image_ok | ~audit.trimap_ok | (audit.fg_frac < 0.02)]
problems

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from bgbias import compose

def show_overlays(ids, cols=6, title=None):
    rows = int(np.ceil(len(ids) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    for ax, image_id in zip(axes.flat, ids):
        img = np.asarray(compose.load_image(PET_ROOT / "images" / f"{image_id}.jpg"), dtype=float) / 255
        tri = compose.load_trimap(PET_ROOT / "annotations" / "trimaps" / f"{image_id}.png")
        overlay = img.copy()
        overlay[tri == 2] *= 0.35                         # background dimmed
        overlay[tri == 3] = [1.0, 0.1, 0.1]               # border in red
        ax.imshow(overlay); ax.set_title(image_id, fontsize=8)
    for ax in axes.flat: ax.axis("off")
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()

rng = np.random.default_rng(0)
show_overlays(rng.choice(all_ids, 24, replace=False), title="Random masks: background dimmed, border red")

In [ ]:
# Smallest pet regions — most likely to break after background replacement.
show_overlays(audit.nsmallest(12, "fg_frac").image_id.tolist(), title="Smallest foreground fraction")

### Граница trimap: включать в животное или нет

Слева — только класс 1, справа — класс 1 + граница (3). Фон ярко-зелёный, чтобы были видны края.
Решение записываем в `PLAN.md` (по умолчанию `include_border=True`, `blur_sigma=1.5`).

In [ ]:
green = (40, 200, 60)
ids = rng.choice(all_ids, 6, replace=False)
fig, axes = plt.subplots(len(ids), 3, figsize=(10, 3.2 * len(ids)))
for row, image_id in zip(axes, ids):
    img = compose.load_image(PET_ROOT / "images" / f"{image_id}.jpg")
    tri = compose.load_trimap(PET_ROOT / "annotations" / "trimaps" / f"{image_id}.png")
    row[0].imshow(img); row[0].set_title(image_id, fontsize=8)
    for ax, border in zip(row[1:], (False, True)):
        alpha = compose.foreground_alpha(tri, include_border=border, blur_sigma=1.5)
        ax.imshow(compose.foreground_only(img, alpha, color=green))
        ax.set_title(f"include_border={border}", fontsize=8)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Пулы фонов (Places365)

In [ ]:
from bgbias import backgrounds as B

index = B.load_places_index(PLACES_ROOT)
pools = B.make_background_pools(index, fractions=(0.6, 0.15, 0.25), seed=SEED)
pools.to_csv(Path(REPO_DIR) / "manifests" / "background_pools.csv", index=False)

excluded = sorted(set(index.category) - set(pools.category))
print(f"{index.category.nunique()} categories, excluded {len(excluded)}:", excluded)
pools.groupby("pool").agg(images=("bg_path", "size"), categories=("category", "nunique"))

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for row, pool in zip(axes, B.POOLS):
    sample = pools[pools.pool == pool].sample(8, random_state=0)
    for ax, (_, r) in zip(row, sample.iterrows()):
        ax.imshow(compose.load_image(PLACES_ROOT / r.bg_path))
        ax.set_title(f"{pool}: {r.category}", fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Фиксированные фоны для val и test

In [ ]:
manifests = {
    "val": B.make_eval_manifest(splits["val"], pools, "val", seed=SEED),
    "test": B.make_eval_manifest(splits["test"], pools, "test", seed=SEED + 1),
}
for name, m in manifests.items():
    m.to_csv(Path(REPO_DIR) / "manifests" / f"{name}_swapped.csv", index=False)
    print(f"{name}: {len(m)} images, {m.bg_path.nunique()} unique backgrounds, {m.bg_category.nunique()} categories")
manifests["test"].head()

## 5. Галерея примеров

In [ ]:
from bgbias.datasets import PetBackgroundDataset

def make_gallery(df, manifest, n=24, seed=0, path=None):
    sample = df.sample(n, random_state=seed).reset_index(drop=True)
    views = {c: PetBackgroundDataset(sample, PET_ROOT, condition=c, places_root=PLACES_ROOT, manifest=manifest)
             for c in ("original", "swapped", "fgonly", "bgonly")}
    cols = ["original", "trimap", "swapped", "fgonly", "bgonly"]
    fig, axes = plt.subplots(n, len(cols), figsize=(3 * len(cols), 2.8 * n))
    for i in range(n):
        image_id = sample.at[i, "image_id"]
        tri = compose.load_trimap(PET_ROOT / "annotations" / "trimaps" / f"{image_id}.png")
        for j, col in enumerate(cols):
            ax = axes[i, j]
            ax.imshow(tri, cmap="viridis") if col == "trimap" else ax.imshow(views[col].render(i))
            ax.axis("off")
            if i == 0: ax.set_title(col, fontsize=12)
        if i > 0: axes[i, 0].set_title(image_id, fontsize=8)
    plt.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=80, bbox_inches="tight")
    plt.show()

gallery_path = Path(REPO_DIR) / "report" / "figures" / "week1_gallery.jpg"
make_gallery(splits["test"], manifests["test"], n=24, path=gallery_path)
print("saved", gallery_path)

## 6. Smoke test: DataLoader с заменой фона p = 0.5

In [ ]:
import time, torch
from torch.utils.data import DataLoader
from bgbias.datasets import build_transform, IMAGENET_MEAN, IMAGENET_STD
from bgbias.utils import seed_everything

seed_everything(0)
train_bg = pools.loc[pools.pool == "train", "bg_path"].tolist()
train_ds = PetBackgroundDataset(splits["train"], PET_ROOT, condition="original", transform=build_transform(train=True),
                                places_root=PLACES_ROOT, bg_pool=train_bg, p_swap=0.5)
loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2,
                    generator=torch.Generator().manual_seed(0))

t0 = time.time()
for i, (x, y) in enumerate(loader):
    if i == 4: break
print(f"batch {tuple(x.shape)}, labels {tuple(y.shape)}, {(time.time() - t0) / 5:.2f} s/batch")

breed_of = dict(zip(splits["train"].label, splits["train"].breed))
mean, std = torch.tensor(IMAGENET_MEAN)[:, None, None], torch.tensor(IMAGENET_STD)[:, None, None]
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for ax, img, label in zip(axes.flat, x[:16], y[:16]):
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(breed_of[label.item()], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Сохранение

Все файлы уже лежат в `/content/dl_project`. Копия идёт в `MyDrive/bgbias/week1/`.

Чтобы отправить их в GitHub (ветка `data/week1`): добавь GitHub token с правом `repo` в **Colab Secrets** (значок ключа слева) под именем `GITHUB_TOKEN`, поставь `PUSH = True` и запусти ячейку.

In [ ]:
import shutil
OUTPUTS = ["splits", "manifests", "results/data_audit.csv", "report/figures/week1_gallery.jpg"]
for rel in OUTPUTS:
    src, dst = Path(REPO_DIR) / rel, OUT_DRIVE / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True) if src.is_dir() else shutil.copy(src, dst)
print("copied to", OUT_DRIVE)

PUSH = False
GIT_NAME, GIT_EMAIL = "Colab", "colab@users.noreply.github.com"   # put your own
if PUSH:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    def git(*args):
        r = subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True)
        print((r.stdout + r.stderr).replace(token, "***"))
        r.check_returncode()
    git("checkout", "-B", "data/week1")
    git("add", *OUTPUTS)
    git("-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}",
        "commit", "-m", "Week 1: splits, background pools, eval manifests, data audit, gallery")
    git("push", f"https://{token}@github.com/mundmina/dl_project.git", "HEAD:data/week1")